# I6. Calibration

The sixth notebook of the introductory track ({doc}`../index`). It calibrates
A1 small with `rang-core flag`, `calibrate` and `apply`, the way CARACal's
continuum chain does, and checks every solved term against the truth that
wrote the data (I4). About a minute.

**Learning goals.** After this notebook you can

- list the steps of cross-calibration in order: model of the flux
  calibrator, delay K, gain G, bandpass B, flux scale F, apply;
- say what a solution interval and a reference antenna are;
- read CASA calibration tables written by rang and check them against a
  known truth in the right gauge;
- check calibrated data on a calibrator against its model, and its
  residuals against the noise, per antenna;
- say what self-calibration is and where it stops on these data.

In [ ]:
%matplotlib inline
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rang import core_ms, fluxstd
from rang import simtruth as st
from rang.notebook import data

os.environ.setdefault("RAYON_NUM_THREADS", "2")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "ms-to-image").is_dir()), None)
WORK = data.default_cache_dir() / "intro-i6"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
COMMANDS = {"rang-core": [sys.executable, "-m", "rang.core_cli"], "python": [sys.executable]}


def sh(line, tail=None):
    # Run one command line in WORK as a shell would; print it and its output.
    print("$", line)
    argv = shlex.split(line)
    p = subprocess.run(COMMANDS.get(argv[0], [argv[0]]) + argv[1:], cwd=WORK, capture_output=True, text=True, encoding="utf-8")
    lines = (p.stdout + p.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    assert p.returncode == 0, f"exit {p.returncode}"


sh("python -m rang.simtruth_ms generate small . --workers 2", tail=1)
for name in ("config.yaml", "strategy.yaml"):  # E4's configuration (notebook 00)
    shutil.copy(ROOT / "examples" / "ms-to-image" / name, WORK / name)
MS = WORK / "a1-small.ms"
d = st.design("small")
z = np.load(WORK / "truth.npz")
freq = z["freq_hz"]

## 1. The plan

An observation interleaves the target with calibrators whose sky is known
(I1). A1 small has four fields:

In [ ]:
times, scans, fields, within = st.schedule(d)
pd.DataFrame(
    [
        {"field": name, "role": role, "scans": int(len(set(scans[fields == i]))), "integrations": int((fields == i).sum())}
        for i, (name, _, _, role, _) in enumerate(st.FIELDS)
    ]
)

The calibration section of E4's configuration, as CARACal runs it:

In [ ]:
text = (WORK / "config.yaml").read_text(encoding="utf-8")
print(text[text.index("calibrate:") : text.index("# One multi-frequency")])

In order:

1. **Model.** Write the flux calibrator's sky model to MODEL_DATA (CASA's
   `setjy`, Stevens-Reynolds 2016, I1).
2. **K**, the delays (I2, I4), on the flux calibrator: one per antenna and
   hand, solution interval `inf` (one per scan).
3. **G**, the gains, with K applied: amplitude and phase per scan.
4. **B**, the bandpass, with K and G applied: one complex value per channel,
   scans combined. `fillgaps: 70` interpolates over up to 70 flagged
   channels.
5. **Gsec** on the gain calibrator, near the target in the sky, observed
   between target scans. Its flux is not known in advance.
6. **F**, the flux scale (CASA `fluxscale`): compares Gsec's amplitudes with
   the flux calibrator's to give the gain calibrator's flux, and rescales the
   gains to it.
7. **Apply** to CORRECTED_DATA: the flux calibrator gets its own K, G, B; the
   gain calibrator and the target get K and B from the flux calibrator and F
   from the nearest gain-calibrator scan, interpolated in time.

Each solve fixes the phase of the **reference antenna** (`refant: m000`) to
zero (I4). `uvrange: ">150"` leaves out baselines shorter than 150 m, where
RFI is worst (I5). `minsnr` and `minblperant` flag solutions with too little
signal or too few baselines.

## 2. Run it

In [ ]:
sh("rang-core flag config.yaml a1-small.ms", tail=2)
sh("rang-core calibrate config.yaml a1-small.ms", tail=2)
sh("rang-core apply config.yaml a1-small.ms", tail=2)
CAL = WORK / "a1-small.ms.rang-run" / "calibrate"
summary = json.loads((CAL / "summary.json").read_text(encoding="utf-8"))
pd.DataFrame(
    [
        {"table": s["name"], "type": s["type"], "rows": s["rows"], "converged": s["solutions"][0]["converged_fraction"] if "solutions" in s else "–"}
        for s in summary["steps"]
    ]
)

Each table is a CASA calibration table: CASA's `applycal` and `plotms`
read them as they are. The truth is in CASA's gauge for m000
(`truth.npz`, the `gauge0_` arrays; I4).

## 3. K against the truth

In [ ]:
def table(name, *columns):
    with core_ms.open_table(str(CAL / name)) as t:
        return [t.getcol(c) for c in columns]


k_sol, k_ant = table("K", "FPARAM", "ANTENNA1")
k_true = z["gauge0_tau_ns"][k_ant]
err = np.abs(k_sol[:, 0, :] - k_true)
print(f"{len(k_sol)} K solutions (16 antennas × 2 scans); true |tau| up to {np.abs(z['gauge0_tau_ns']).max():.1f} ns")
print(f"|K − truth|: median {np.median(err) * 1e3:.1f} ps, max {err.max() * 1e3:.1f} ps")

## 4. B against the truth

G, solved per scan before B, holds a constant phase and amplitude per
antenna, so B matches the truth up to that constant; and B absorbs what K
left over, a small phase slope. Compare K·B, solved against true, each
normalised by its mean over the band:

In [ ]:
b_sol, b_ant, b_flag = table("B", "CPARAM", "ANTENNA1", "FLAG")
nu0 = float(z["nu0_hz"])


def k_phase(tau_ns):  # (antenna, hand) delays -> (antenna, channel, hand) K
    return np.exp(2j * np.pi * tau_ns[:, None, :] * 1e-9 * (freq - nu0)[None, :, None])


k_mean = np.array([k_sol[k_ant == a, 0, :].mean(axis=0) for a in b_ant])  # the two scans' K
b_sol = b_sol * k_phase(k_mean)
b_true = z["gauge0_bandpass"][b_ant] * k_phase(z["gauge0_tau_ns"][b_ant])


def normalise(b, flag=None):
    b = np.ma.masked_array(b, flag if flag is not None else np.zeros(b.shape, bool))
    # mean amplitude 1 and mean phase 0 per antenna and hand
    mean = np.ma.mean(b, axis=1, keepdims=True)
    return b / np.ma.mean(np.ma.abs(b), axis=1, keepdims=True) * np.ma.exp(-1j * np.ma.angle(mean))


bs, bt = normalise(b_sol, b_flag), normalise(b_true)
amp_err = np.ma.abs(np.ma.abs(bs) - np.abs(bt))
ph_err = np.degrees(np.ma.abs(np.ma.angle(bs / bt)))
print(f"K·B amplitude error: median {np.ma.median(amp_err):.4f}, 99th percentile {np.percentile(amp_err.compressed(), 99):.4f}")
print(f"K·B phase error: median {np.ma.median(ph_err):.2f} deg, 99th percentile {np.percentile(ph_err.compressed(), 99):.2f} deg")
print(f"flagged B solutions: {b_flag.mean():.1%}")
a = 3
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for h, hand in enumerate("XY"):
    axes[0].plot(freq / 1e6, np.ma.abs(bs[a, :, h]), "o", ms=3, label=f"solved, {hand}")
    axes[0].plot(freq / 1e6, np.abs(bt[a, :, h]), "-", lw=1, label=f"truth, {hand}")
    axes[1].plot(freq / 1e6, np.degrees(np.ma.angle(bs[a, :, h])), "o", ms=3)
    axes[1].plot(freq / 1e6, np.degrees(np.angle(bt[a, :, h])), "-", lw=1)
axes[0].set(xlabel="frequency (MHz)", ylabel="|KB| / mean", title=f"K·B of m{4 * b_ant[a]:03d}")
axes[1].set(xlabel="frequency (MHz)", ylabel="phase − mean (deg)")
axes[0].legend(fontsize=8)
plt.show()

## 5. The flux scale

`fluxscale` transfers the flux calibrator's scale to the gain calibrator.
The flux it found, and the truth A1 wrote:

In [ ]:
fs = next(s for s in summary["steps"] if s["type"] == "F")["fluxscale"]["SIM-GCAL"]
true_spectrum = st.calibrator_stokes(d)[2, :, 0]
print(f"fluxscale: {fs['fitted_flux_jy']:.3f} Jy at {fs['reference_frequency_hz'] / 1e6:.0f} MHz (spectral index {fs['spectral_index']}, not fitted)")
print(f"truth: {np.interp(fs['reference_frequency_hz'], freq, true_spectrum):.3f} Jy at that frequency, {true_spectrum.mean():.3f} Jy averaged over the band "
      f"(model {st.GCAL['fluxdensity']} Jy at {st.GCAL['reffreq_hz'] / 1e6:.0f} MHz, spectral index {st.GCAL['spix'][0]})")

`fluxscale` averages over the band and, as configured here, fits no
spectral index, so its number is to be compared with the band average.

## 6. Calibrated data against the model

On the gain calibrator, CORRECTED_DATA should equal its model (an
unpolarised point source at the phase centre: XX = YY = I, real) plus noise.
Averaged over baselines and time, per channel:

In [ ]:
with core_ms.open_table(str(MS)) as t:
    field = t.getcol("FIELD_ID")
    rows = np.flatnonzero(field == 2)
    cols = {c: t.getcol(c, int(rows[0]), int(rows[-1] - rows[0] + 1)) for c in ("DATA", "CORRECTED_DATA", "FLAG")}
sel = field[rows[0] : rows[-1] + 1] == 2
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for c, style in (("DATA", "C0."), ("CORRECTED_DATA", "C1.")):
    v = np.ma.masked_array(cols[c][sel][..., [0, 3]], cols["FLAG"][sel][..., [0, 3]]).mean(axis=(0, 2))
    axes[0].plot(freq / 1e6, np.ma.abs(v), style, label=c)
    axes[1].plot(freq / 1e6, np.degrees(np.ma.angle(v)), style)
axes[0].plot(freq / 1e6, true_spectrum, "k-", label="true model")
axes[0].set(xlabel="frequency (MHz)", ylabel="|mean of XX, YY| (Jy)", title="SIM-GCAL")
axes[1].set(xlabel="frequency (MHz)", ylabel="phase (deg)")
axes[0].legend()
plt.show()
v = np.ma.masked_array(cols["CORRECTED_DATA"][sel][..., [0, 3]], cols["FLAG"][sel][..., [0, 3]]).mean(axis=(0, 2))
ratio = np.ma.abs(v) / true_spectrum
print(f"|corrected| / truth over unflagged channels: median {np.ma.median(ratio):.3f}, range {ratio.min():.3f}-{ratio.max():.3f}; "
      f"phase rms {np.degrees(np.sqrt(np.ma.mean(np.ma.angle(v) ** 2))):.2f} deg")

The target's CORRECTED_DATA is what notebook I7 images.

## 7. Residuals against the noise

The plot above averages over baselines and time, which hides errors that
average out. The sharper test is per visibility: the **residual**,
corrected data minus the model, should be noise. Its scatter is compared
with that of the same cells corrected with the *true* Jones terms (I4),
which carry exactly the same noise and RFI: a ratio of 1 means the
calibration errors are below the noise. Per antenna, over the baselines
that include it, the same ratio points at the antenna whose solutions are
worst.

In [ ]:
z_truth = {k.removeprefix("raw_"): z[k] for k in z.files if k.startswith("raw_")}
with core_ms.open_table(str(MS)) as t:
    a1, a2 = t.getcol("ANTENNA1"), t.getcol("ANTENNA2")
nbl = len(d.antennas) * (len(d.antennas) - 1) // 2
span = slice(int(rows[0]), int(rows[-1]) + 1)
g_rows = st.Rows((np.arange(len(a1)) // nbl)[span][sel], a1[span][sel], a2[span][sel])
true_corrected = st.correct(cols["DATA"][sel], z_truth, d, g_rows, z["parallactic_angle"])
model = np.zeros_like(true_corrected)
model[..., 0] = model[..., 3] = true_spectrum[None, :]  # unpolarised point source, XX = YY = I
mask = cols["FLAG"][sel][..., [0, 3]]


def scatter(v, which=slice(None), part="real"):
    r = np.ma.masked_array((v - model)[..., [0, 3]], mask)[which]
    return float(np.ma.sqrt(np.ma.mean(getattr(r, part) ** 2)))


sigma = float(json.loads((WORK / "truth.json").read_text(encoding="utf-8"))["design"]["sigma_jy"])
print(f"sigma per real part in DATA: {sigma * 1e3:.1f} mJy")
for part in ("real", "imag"):
    cal, ideal = scatter(cols["CORRECTED_DATA"][sel], part=part), scatter(true_corrected, part=part)
    print(f"rms of the residual's {part} part: calibrated {cal * 1e3:.1f} mJy, truth-corrected {ideal * 1e3:.1f} mJy (ratio {cal / ideal:.3f})")
per_antenna = pd.DataFrame(
    [
        {"antenna": f"m{d.antennas[a]:03d}",
         "calibrated / truth-corrected": round(scatter(cols["CORRECTED_DATA"][sel], (g_rows.a1 == a) | (g_rows.a2 == a)) / scatter(true_corrected, (g_rows.a1 == a) | (g_rows.a2 == a)), 3)}
        for a in range(len(d.antennas))
    ]
).set_index("antenna")
per_antenna.T

Read the two lines together. The truth-corrected scatter is well above σ
in both parts: correcting divides the data by the gains and the bandpass,
which divides the noise by them too, and the model is real, so the
imaginary part is noise alone. The calibrated residual exceeds it mostly
in the imaginary part, which is a phase error, and unevenly across
antennas (the table). The gain calibrator's true gains are constant within
each scan, so this is not time variation that per-scan solutions miss.
This notebook does not locate the cause; exercise 6 starts on it.

## 8. Self-calibration

The calibrators fix the gains at their times and directions. The target's
own gains differ (A1 adds a slowly varying gain S on the target only, I4).
**Self-calibration** uses the target itself: image it, take the
deconvolved model as M, solve for gains against it, apply, image again.
`rang-core selfcal` runs CARACal's rounds. On A1, the loop without a mask
does not meet its truth limits: the unmasked CLEAN model absorbs part of the
gain errors ({gh}`research/c5-selfcal.md`, {gh}`research/c5b-selfcal-masks.md`).
This is recorded as a known limit, not hidden.

## Exercises

1. Change G's `solint` from `inf` to `16s` (two integrations). How many G
   rows do you get, and how do they compare with `truth.npz`'s
   `gauge0_gain` at those times?
2. Set `refant: m020` and repeat section 3 with the truth in that gauge
   (`st.casa_gauge(raw truth, 5, freq, nu0)`, I4).
3. Remove the flag stage (`rang-core calibrate` straight after generation).
   Which table changes most, and does the flux scale move?
4. Leave B out of the gain calibrator's and target's `apply` tables. What do
   sections 6 and 7 show?
5. Corrupt one antenna's K table entry by 0.5 ns (write FPARAM with
   `core_ms_write`) and apply again. Does section 7's per-antenna table
   find it?
6. For the antenna with the largest ratio in section 7, compare its K and
   K·B phase errors (sections 3 and 4) with the others'. Is its excess a
   delay (phase linear in frequency) or a bandpass error?

## Where next

- {doc}`00-ms-to-image` runs these stages and the image in one
  `rang-core run`; {doc}`../../cli` lists every option.
- {doc}`i7-imaging-and-deconvolution` images the target.
- Notebook {doc}`03-gains-and-tv` solves gains on real data and looks at
  the gauge and solution intervals; {doc}`../../concepts/gains`.